# v0.3 S4-NQ — reranker (E5), full corpus (E9), HNSW (E6), mechanism tests (E12), repeated cost (E7)
**How to run:** Settings → Accelerator **GPU T4 x2**, Internet **On**. Then **Save Version → Save & Run All (Commit)**. It runs unattended (about 5–6 h). If a step fails, the notebook stops with `FAILED` — send me the log. When it finishes, open the version's **Output** tab, download `v03_s4nq_results.zip` and send it to me.

**Input:** Add Input → **Your Work → Notebooks** → the v0.2 **Main S1** notebook from 27 Sep (its output has `main_s1_results.zip`), the same input as notebook 3.

In [ ]:
# Cell 0 — helper: run a shell command, stream its output, STOP the notebook if it fails
import os, subprocess
def sh(cmd):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {cmd}")

In [ ]:
# Cell 0b — run two commands at once (one per GPU); STOP if either fails
import threading
def sh_par(*cmds):
    errs = []
    def run(c):
        try:
            sh(c)
        except Exception as e:
            errs.append(str(e))
    ts = [threading.Thread(target=run, args=(c,)) for c in cmds]
    [t.start() for t in ts]; [t.join() for t in ts]
    if errs:
        raise RuntimeError("; ".join(errs))

In [ ]:
# Cell 0d — a failing step is recorded and the notebook continues, so finished results are always packaged
GEN_ERR = ""
def soft(*cmds, par=False):
    global GEN_ERR
    try:
        sh_par(*cmds) if par else [sh(c) for c in cmds]
    except Exception as e:
        GEN_ERR += ("; " if GEN_ERR else "") + str(e); print("STEP FAILED (recorded, continuing):", e)

In [ ]:
# Cell 0c — the code of commit a86b2d5 as base64 tar.gz (do not edit; Cell 1 checks it)
CODE = "..."

In [ ]:
# Cell 1 — hardware, code frozen at a86b2d5 (embedded in Cell 0c), pinned environment, check, tests
sh("nvidia-smi --query-gpu=index,name,driver_version,memory.total --format=csv || true")
sh("df -h / /tmp /kaggle/working && free -g && nproc")
os.chdir("/kaggle/working")
try:                                   # Hugging Face token from Kaggle Secrets (Add-ons -> Secrets -> HF_TOKEN)
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN"); print("HF_TOKEN loaded")
except Exception as e:
    print("HF_TOKEN not available (public downloads still work):", type(e).__name__)
# the GitHub repo is private: the code of commit a86b2d5dec8d53549c3f7146e6a20ff5d0a3a5e3 is embedded in Cell 0c (no dataset, no clone);
# STOP unless the extracted files reproduce that commit's git tree exactly (paper_inputs, paper_outputs left out)
import base64, io, shutil, tarfile
shutil.rmtree("rag", ignore_errors=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE))).extractall("rag")
os.chdir("/kaggle/working/rag")
sh("git init -q && git add -A -f")
sh("test \"$(git write-tree)\" = a554fcd1a8c6cc4a378cb3ff7ecd933314f2ce35 || (echo WRONG CODE; exit 1)")
sh("git -c user.name=kaggle -c user.email=kaggle@local commit -qm 'code a86b2d5dec8d53549c3f7146e6a20ff5d0a3a5e3' && echo CODE OK a86b2d5")
sh("pip install -q -r requirements-v03-retrieval.txt")
sh("pip install -q -r requirements-v03-generation.txt")
sh("pip install -q --force-reinstall --no-deps torchaudio==2.11.0")
sh("python -c \"import torch, torchaudio, torchvision, vllm, transformers; print(torch.__version__, torch.version.cuda, torchaudio.__version__, torchvision.__version__, vllm.__version__, transformers.__version__); assert torch.cuda.is_available() and torchaudio.__version__.endswith(torch.version.cuda.replace(\x27.\x27, \x27\x27)), \x27torch/torchaudio CUDA mismatch\x27\"")

sh("python -m pytest -q -p no:cacheprovider 2>&1 | tail -3")

In [ ]:
# Cell 2 — restore the v0.2 S1 output FIRST, then environment record, then NQ corpus text + qrels (v0.2 pinned revisions)
sh("python -m src.restore --name main_s1_results")
sh("python -m src.check_env --lock requirements-generation.lock --report-only results/raw/check_env_v03_S4nq.json")
sh("python -m src.env_report --session v03_S4nq --out results/raw/env_v03_S4nq.json")
sh("python -m src.data.download --config configs/v03_nq_qwen.yaml --only nq_corpus,nq_qrels")

In [ ]:
# Cell 3 — in parallel (~2 h): GPU 0 = E9 full-corpus encoding (R1, R2, R5; 2.68M passages);
# GPU 1 = E5 rerank R0/R5 top 50 -> X0/X5, E12 contexts, reader (Qwen) on X0, X5 (2,255) and M1, M3, M5, MR, MG (500)
soft("CUDA_VISIBLE_DEVICES=0 python -m src.retrieval.encode --config configs/v03_nq_qwen.yaml --models R1,R2,R5 --corpus full",
     "CUDA_VISIBLE_DEVICES=1 python -m src.retrieval.rerank --config configs/v03_nq_qwen.yaml --first R0,R5 --depth 50 && python -m src.generation.mechanism --config configs/v03_nq_qwen.yaml --n 500 && CUDA_VISIBLE_DEVICES=1 python -m src.generation.run --config configs/v03_nq_qwen.yaml --conditions X0,X5 --corpus main --n 2255 --k 5 --auto-eager-retry && CUDA_VISIBLE_DEVICES=1 python -m src.generation.run --config configs/v03_nq_qwen.yaml --conditions M1,M3,M5,MR,MG --corpus main --n 500 --k 5 --auto-eager-retry", par=True)

In [ ]:
# Cell 4 — CPU: E9 exact search on the full corpus (all judged queries), retrieval summaries, E6 HNSW
soft(*[f"python -m src.retrieval.search --config configs/v03_nq_qwen.yaml --model {m} --corpus full --queries judged" for m in ("R1", "R2", "R5")])
soft("python -m src.analysis.retrieval_summary --config configs/v03_nq_qwen.yaml --corpus full --retrievers R0,R1,R2,R5 --out summary/v03_nq_full_summary.json",
     "python -m src.analysis.retrieval_summary --config configs/v03_nq_qwen.yaml --corpus main --retrievers R0,R1,R2,R3,R4,R5,R6,X0,X5 --out summary/v03_nq_main_summary.json")
soft("python -m src.retrieval.hnsw --config configs/v03_nq_qwen.yaml --models R1,R2,R5")
sh("rm -rf data/embeddings /tmp/models; df -h /kaggle/working")

In [ ]:
# Cell 5 — score answers (EM, F1, answer containment) for X0, X5, M1..MG
soft("python -m src.evaluation.answer --config configs/v03_nq_qwen.yaml --prefix v03_nq_qwen")

In [ ]:
# Cell 6 — E7: five repetitions of the cost measurements, alone at the end of the session (~2.5 h)
soft("CUDA_VISIBLE_DEVICES=0 python -m src.retrieval.throughput --config configs/v03_nq_qwen.yaml --models R1,R2,R3,R5,R4,R6 --n 20000 --batch 128", "python -m src.retrieval.bm25 --config configs/v03_nq_qwen.yaml --corpus main --queries judged --latency")   # repetition 1
soft("CUDA_VISIBLE_DEVICES=0 python -m src.retrieval.throughput --config configs/v03_nq_qwen.yaml --models R1,R2,R3,R5,R4,R6 --n 20000 --batch 128", "python -m src.retrieval.bm25 --config configs/v03_nq_qwen.yaml --corpus main --queries judged --latency")   # repetition 2
soft("CUDA_VISIBLE_DEVICES=0 python -m src.retrieval.throughput --config configs/v03_nq_qwen.yaml --models R1,R2,R3,R5,R4,R6 --n 20000 --batch 128", "python -m src.retrieval.bm25 --config configs/v03_nq_qwen.yaml --corpus main --queries judged --latency")   # repetition 3
soft("CUDA_VISIBLE_DEVICES=0 python -m src.retrieval.throughput --config configs/v03_nq_qwen.yaml --models R1,R2,R3,R5,R4,R6 --n 20000 --batch 128", "python -m src.retrieval.bm25 --config configs/v03_nq_qwen.yaml --corpus main --queries judged --latency")   # repetition 4
soft("CUDA_VISIBLE_DEVICES=0 python -m src.retrieval.throughput --config configs/v03_nq_qwen.yaml --models R1,R2,R3,R5,R4,R6 --n 20000 --batch 128", "python -m src.retrieval.bm25 --config configs/v03_nq_qwen.yaml --corpus main --queries judged --latency")   # repetition 5

In [ ]:
# Last cell — package: everything EXCEPT the large run files (they stay in the earlier output) -> v03_s4nq_results.zip
# small (about 20-40 MB): send this file to Claude
import zipfile
from pathlib import Path
n = 0
with zipfile.ZipFile("/kaggle/working/v03_s4nq_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for k in ['results/raw', 'results/processed', 'summary', 'configs/resolved_revisions.json', 'logs']:
        for p in ([Path(k)] if Path(k).is_file() else sorted(Path(k).rglob("*"))):
            if p.is_file() and ("/runs/" not in p.as_posix() or p.name.startswith(('X', 'M'))) and (not k.startswith("data/") or p.name.startswith(())):
                z.write(p); n += 1
print(n, "files"); sh("ls -la /kaggle/working/v03_s4nq_results.zip")
if GEN_ERR:
    raise RuntimeError("FAILED (packaged anyway): " + GEN_ERR)